# 实验结果分析

本notebook用于分析和可视化实验运行结果。

## 1. 导入必要的库

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

# 设置显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# 设置绘图风格
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("库导入完成")

## 2. 加载实验数据

In [ ]:
# 导入实验跟踪器
import sys
sys.path.insert(0, '.')
from experiment_tracker import ExperimentTracker

# 创建实验跟踪器
tracker = ExperimentTracker()

# 加载实验历史
df = tracker.get_history_df()

if df.empty:
    print("没有找到实验记录。请先运行一些实验。")
    print("\n使用方法:")
    print("  python run_experiments.py --list")
    print("  python run_experiments.py --experiment standard")
    print("  python run_experiments.py --all")
else:
    print(f"加载了 {len(df)} 条实验记录")
    print(f"\n数据列: {list(df.columns)}")
    df.head()

## 3. 实验概览

In [ ]:
# 打印实验摘要
tracker.print_summary()

## 4. 查看最佳实验

In [ ]:
# 获取验证集R²最高的前10个实验
if not df.empty:
    best_experiments = tracker.get_best_experiments('valid_r2', n=10)
    
    print("验证集R²最高的10个实验:")
    print("="*80)
    
    # 只显示关键列
    key_cols = ['experiment_id', 'valid_r2', 'train_r2', 'learning_rate', 'max_depth', 
                'num_leaves', 'n_estimators', 'timestamp']
    available_cols = [col for col in key_cols if col in best_experiments.columns]
    
    display(best_experiments[available_cols])

## 5. 参数影响分析

分析不同参数对模型性能的影响

In [ ]:
def plot_parameter_effect(df, param_name, metric='valid_r2'):
    """
    绘制参数对性能的影响
    
    参数:
        df: 实验数据
        param_name: 参数名
        metric: 评估指标
    """
    if param_name not in df.columns:
        print(f"参数 '{param_name}' 不在数据中")
        return
    
    # 按参数分组
    grouped = df.groupby(param_name)[metric].agg(['mean', 'std', 'count'])
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 左图：性能 vs 参数
    axes[0].errorbar(grouped.index, grouped['mean'], yerr=grouped['std'],
                     marker='o', linestyle='-', capsize=5, capthick=2)
    axes[0].set_xlabel(param_name)
    axes[0].set_ylabel(metric)
    axes[0].set_title(f'{metric} vs {param_name}')
    axes[0].grid(True, alpha=0.3)
    
    # 右图：实验数量
    axes[1].bar(grouped.index, grouped['count'])
    axes[1].set_xlabel(param_name)
    axes[1].set_ylabel('实验数量')
    axes[1].set_title(f'每个{param_name}值的实验数量')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 打印统计
    print(f"\n{param_name} 对 {metric} 的影响:")
    print("="*50)
    print(grouped)

# 分析关键参数
if not df.empty:
    params_to_analyze = ['learning_rate', 'max_depth', 'num_leaves', 'n_estimators']
    
    for param in params_to_analyze:
        if param in df.columns:
            plot_parameter_effect(df, param)

## 6. 参数交互效应

查看两个参数的组合效果

In [ ]:
def plot_parameter_interaction(df, param1, param2, metric='valid_r2'):
    """
    绘制两个参数的交互效应热图
    
    参数:
        df: 实验数据
        param1: 第一个参数
        param2: 第二个参数
        metric: 评估指标
    """
    if param1 not in df.columns or param2 not in df.columns:
        print(f"参数不在数据中")
        return
    
    # 创建透视表
    pivot = df.pivot_table(values=metric, index=param1, columns=param2, aggfunc='mean')
    
    # 绘制热图
    plt.figure(figsize=(10, 8))
    sns.heatmap(pivot, annot=True, fmt='.4f', cmap='RdYlGn', 
                cbar_kws={'label': metric})
    plt.title(f'{metric}: {param1} vs {param2}')
    plt.tight_layout()
    plt.show()

# 分析参数交互
if not df.empty:
    # 学习率 vs 最大深度
    if 'learning_rate' in df.columns and 'max_depth' in df.columns:
        plot_parameter_interaction(df, 'learning_rate', 'max_depth')

## 7. 过拟合分析

比较训练集和验证集的性能，识别过拟合

In [ ]:
if not df.empty and 'train_r2' in df.columns and 'valid_r2' in df.columns:
    # 计算过拟合程度
    df['overfitting'] = df['train_r2'] - df['valid_r2']
    
    # 绘制散点图
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 训练集 vs 验证集
    axes[0].scatter(df['train_r2'], df['valid_r2'], alpha=0.6)
    axes[0].plot([0, 1], [0, 1], 'r--', label='理想情况')
    axes[0].set_xlabel('训练集 R²')
    axes[0].set_ylabel('验证集 R²')
    axes[0].set_title('训练集 vs 验证集性能')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # 过拟合分布
    axes[1].hist(df['overfitting'], bins=20, edgecolor='black')
    axes[1].axvline(df['overfitting'].mean(), color='red', 
                   linestyle='--', label=f'平均: {df["overfitting"].mean():.4f}')
    axes[1].set_xlabel('过拟合程度 (train_r2 - valid_r2)')
    axes[1].set_ylabel('实验数量')
    axes[1].set_title('过拟合分布')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 找出过拟合最严重的实验
    print("\n过拟合最严重的5个实验:")
    print("="*60)
    worst_overfitting = df.nlargest(5, 'overfitting')[['experiment_id', 'train_r2', 'valid_r2', 'overfitting']]
    display(worst_overfitting)

## 8. 实验时间线

查看实验随时间的进展

In [ ]:
if not df.empty and 'timestamp' in df.columns:
    df['datetime'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values('datetime')
    
    plt.figure(figsize=(14, 6))
    
    if 'valid_r2' in df.columns:
        plt.plot(df.index, df['valid_r2'], marker='o', label='验证集 R²')
    if 'train_r2' in df.columns:
        plt.plot(df.index, df['train_r2'], marker='s', alpha=0.6, label='训练集 R²')
    
    plt.xlabel('实验序号')
    plt.ylabel('R² 得分')
    plt.title('实验性能随时间的变化')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # 标注最佳点
    if 'valid_r2' in df.columns:
        best_idx = df['valid_r2'].idxmax()
        best_val = df.loc[best_idx, 'valid_r2']
        plt.annotate(f'最佳: {best_val:.4f}', 
                    xy=(best_idx, best_val),
                    xytext=(10, 10), textcoords='offset points',
                    bbox=dict(boxstyle='round,pad=0.5', fc='yellow', alpha=0.5),
                    arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))
    
    plt.tight_layout()
    plt.show()

## 9. 对比特定实验

In [ ]:
# 对比多个实验
if not df.empty:
    # 选择验证集R²最高的前5个实验
    best_ids = df.nlargest(5, 'valid_r2')['experiment_id'].tolist()
    
    comparison = tracker.compare_experiments(best_ids)
    
    print("对比验证集R²最高的5个实验:")
    print("="*80)
    
    # 只显示关键列
    key_cols = ['experiment_id', 'valid_r2', 'train_r2', 'learning_rate', 'max_depth', 
                'num_leaves', 'n_estimators']
    available_cols = [col for col in key_cols if col in comparison.columns]
    
    display(comparison[available_cols])

## 10. 导出结果

In [ ]:
# 保存为CSV
tracker.save_to_csv()

# 保存最佳实验配置
if not df.empty:
    best_exp = df.nlargest(1, 'valid_r2').iloc[0]
    
    # 提取参数
    best_config = {}
    param_cols = ['learning_rate', 'max_depth', 'num_leaves', 'min_data_in_leaf', 
                  'n_estimators', 'n_dates_to_load', 'n_symbols_to_use']
    
    for col in param_cols:
        if col in df.columns and col in best_exp.index:
            best_config[col] = best_exp[col]
    
    # 保存为JSON
    config_path = Path('./experiments/best_config.json')
    config_path.parent.mkdir(exist_ok=True)
    
    with open(config_path, 'w') as f:
        json.dump(best_config, f, indent=2)
    
    print(f"\n最佳实验配置已保存到: {config_path}")
    print("\n配置内容:")
    print(json.dumps(best_config, indent=2))

## 11. 快速运行新实验

从这里可以快速运行实验并查看结果

In [ ]:
# 使用示例：运行单个实验
# 在终端中运行以下命令：
# 
# 运行标准配置:
# python run_experiments.py --experiment standard
#
# 运行所有预设实验:
# python run_experiments.py --all
#
# 运行网格搜索:
# python run_experiments.py --grid_search --param learning_rate,max_depth
#
# 列出所有可用实验:
# python run_experiments.py --list

print("在终端中运行以下命令来启动实验:")
print("\n# 运行标准配置:")
print("python run_experiments.py --experiment standard")
print("\n# 运行所有预设实验:")
print("python run_experiments.py --all")
print("\n# 运行网格搜索:")
print("python run_experiments.py --grid_search --param learning_rate,max_depth")